### Analysis of EEG Data with a Linear Mixed Model and Post-Hoc Analysis 
This notebook reads a .csv file with the average coherence values per state and frequency and runs a LME model to determine differences. Post-hoc analysis is run to look at differences between all states and frequencies.  

In [1]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.multicomp as mc

In [2]:
# 1. Load the tidy dataset
df = pd.read_csv("subject_connectivity_stats.csv")

In [3]:
# 2. Filter out 'Overall' so we are strictly comparing the actual frequency bands
df_bands = df[df['Frequency'] != 'Overall'].copy()

print(f"Total rows in analysis: {len(df_bands)}")
print(f"Unique subjects: {df_bands['Subject'].nunique()}")

Total rows in analysis: 192
Unique subjects: 31


In [4]:
# 3. Fit the Linear Mixed-Effects Model
# Formula breakdown:
# Coherence ~ State * Frequency means look at Main Effect of State, 
# Main Effect of Frequency, and their Interaction.
# groups=df_bands['Subject'] specifies that data points are clustered by subject.
model = smf.mixedlm(
    "Coherence ~ C(State) * C(Frequency)", 
    data=df_bands, 
    groups=df_bands['Subject']
)

result = model.fit()

c:\Users\alena\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [5]:
# 4. Print the comprehensive statistical summary
print("\n" + "="*60)
print("          LINEAR MIXED EFFECTS MODEL RESULTS")
print("="*60)
print(result.summary())
print("="*60)


          LINEAR MIXED EFFECTS MODEL RESULTS
                      Mixed Linear Model Regression Results
Model:                      MixedLM         Dependent Variable:         Coherence
No. Observations:           192             Method:                     REML     
No. Groups:                 31              Scale:                      0.0019   
Min. group size:            4               Log-Likelihood:             275.5468 
Max. group size:            8               Converged:                  Yes      
Mean group size:            6.2                                                  
---------------------------------------------------------------------------------
                                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------
Intercept                               0.507    0.013 40.169 0.000  0.482  0.531
C(State)[T.Wake]                       -0.023    0.013 -1.728 0.084 -0.049

In [6]:
# Run a post-hoc pairwise comparison across all combinations of State and Frequency
df_bands['Group'] = df_bands['State'] + "_" + df_bands['Frequency']
comp = mc.MultiComparison(df_bands['Coherence'], df_bands['Group'])
post_hoc_res = comp.tukeyhsd()
print(post_hoc_res)

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
   group1      group2   meandiff p-adj   lower   upper  reject
--------------------------------------------------------------
Sleep_Alpha  Sleep_Beta  -0.0029    1.0 -0.0615  0.0556  False
Sleep_Alpha Sleep_Delta  -0.0048    1.0 -0.0634  0.0537  False
Sleep_Alpha Sleep_Theta   0.0567 0.0655 -0.0019  0.1152  False
Sleep_Alpha  Wake_Alpha  -0.0341 0.5344 -0.0883  0.0201  False
Sleep_Alpha   Wake_Beta  -0.0087 0.9997 -0.0629  0.0455  False
Sleep_Alpha  Wake_Delta  -0.0487 0.1137 -0.1029  0.0055  False
Sleep_Alpha  Wake_Theta   0.0147 0.9911 -0.0395  0.0689  False
 Sleep_Beta Sleep_Delta  -0.0019    1.0 -0.0605  0.0566  False
 Sleep_Beta Sleep_Theta   0.0596 0.0429   0.001  0.1181   True
 Sleep_Beta  Wake_Alpha  -0.0312 0.6462 -0.0854  0.0231  False
 Sleep_Beta   Wake_Beta  -0.0058    1.0   -0.06  0.0484  False
 Sleep_Beta  Wake_Delta  -0.0458 0.1667    -0.1  0.0084  False
 Sleep_Beta  Wake_Theta   0.0176 0.9745 -0.0366  0.0718